# Create z-score files

This notebook creates mean/std files for the paper. Resdidual coefficients are defined in a separated notebook.

In [1]:
import os
import yaml
import numpy as np
import xarray as xr

## ERA5 mean std

In [3]:
# get variable information from data_preprocessing/config
config_name = os.path.realpath('data_config_ERA5.yml')

with open(config_name, 'r') as stream:
    conf = yaml.safe_load(stream)

In [14]:
N_levels = 11

base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/all_in_one/'
ds_example = xr.open_zarr(base_dir+'ERA5_GP_1980.zarr')
level = np.array(ds_example['level'])

In [4]:
varnames = list(conf['zscore'].keys())
varnames = varnames[:-3] # remove save_loc and others
varname_surf = list(set(varnames) - set(['U', 'V', 'T', 'Q']))
varname_upper = ['U', 'V', 'T', 'Q']

# collect computed mean and variance values
# See "qsub_STEP01_compute_mean_std.ipynb"
MEAN_values = {}
STD_values = {}

for varname in varname_surf:
    save_name = conf['zscore']['save_loc'] + '{}_mean_std_{}.npy'.format(
        conf['zscore']['prefix'], varname)
    
    mean_std = np.load(save_name)
    MEAN_values[varname] = mean_std[0]
    STD_values[varname] = mean_std[1]

for varname in varname_upper:

    # -------------------------------------------- #
    # allocate all levels
    mean_std_all_levels = np.empty((2, N_levels))
    mean_std_all_levels[...] = np.nan
    
    for i_level in range(N_levels):
        save_name = conf['zscore']['save_loc'] + '{}_level{}_mean_std_{}.npy'.format(
            conf['zscore']['prefix'], i_level, varname)
        
        mean_std = np.load(save_name)
        mean_std_all_levels[:, i_level] = mean_std

    # -------------------------------------------- #
    # save
    MEAN_values[varname] = np.copy(mean_std_all_levels[0, :])
    STD_values[varname] = np.copy(mean_std_all_levels[1, :])

### Mean file

In [5]:
# ------------------------------------------------------- #
# Initialize dataset
ds_mean_6h = xr.Dataset(coords={"level": level})

for varname, data in MEAN_values.items():
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["level",],
            coords={"level": level},
            name=varname,
        )
        ds_mean_6h[varname] = data_array
    else:
        data_array = xr.DataArray(
            data,
            name=varname,
        )
        ds_mean_6h[varname] = data_array

In [7]:
# ds_mean_6h.to_netcdf('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/ERA5_6h_mean_1980_2019.nc')

In [16]:
ds_mean_6h = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/ERA5_6h_mean_1980_2019.nc')

In [18]:
ds_mean_6h['T'].values

array([292.68453936, 289.9653713 , 285.2367415 , 276.93852369,
       269.37662075, 260.44024499, 248.98519769, 234.04931605,
       217.93857226, 206.35714342, 211.75532093])

### Std file

In [8]:
# ------------------------------------------------------- #
# create xr.DataArray for std

# use the same level coord as mean
ds_std_6h = xr.Dataset(coords={"level": level})

for varname, data in STD_values.items():
    data = np.sqrt(data)
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["level",],
            coords={"level": level},
            name=varname,
        )
        ds_std_6h[varname] = data_array
    else:
        data_array = xr.DataArray(
            data,
            name=varname,
        )
        ds_std_6h[varname] = data_array

In [9]:
# ds_std_6h.to_netcdf('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/ERA5_6h_std_1980_2019.nc')

## WRF mean std

In [4]:
# get variable information from data_preprocessing/config
config_name = os.path.realpath('data_config_WRF.yml')

with open(config_name, 'r') as stream:
    conf = yaml.safe_load(stream)

In [5]:
N_levels = 16

base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/all_in_one/'
ds_example = xr.open_zarr(base_dir+'C404_GP_1980.zarr')
level = np.array(ds_example['bottom_top'])

In [6]:
ds_example['WRF_radar_composite'].min().values

array(0., dtype=float32)

In [7]:
ds_example['WRF_MLCAPE'].min().values

array(0., dtype=float32)

In [8]:
ds_example['WRF_OLR'].min().values

array(77.026855, dtype=float32)

In [5]:
varnames = list(conf['zscore'].keys())
varnames = varnames[:-3] # remove save_loc and others
varname_upper = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q', 'WRF_P']
varname_surf = list(set(varnames) - set(varname_upper))

# collect computed mean and variance values
# See "qsub_STEP01_compute_mean_std.ipynb"
MEAN_values = {}
STD_values = {}

for varname in varname_surf:
    save_name = conf['zscore']['save_loc'] + '{}_mean_std_{}.npy'.format(
        conf['zscore']['prefix'], varname)
    
    mean_std = np.load(save_name)
    MEAN_values[varname] = mean_std[0]
    STD_values[varname] = mean_std[1]

for varname in varname_upper:

    # -------------------------------------------- #
    # allocate all levels
    mean_std_all_levels = np.empty((2, N_levels))
    mean_std_all_levels[...] = np.nan
    
    for i_level in range(N_levels):
        save_name = conf['zscore']['save_loc'] + '{}_level{}_mean_std_{}.npy'.format(
            conf['zscore']['prefix'], i_level, varname)
        
        mean_std = np.load(save_name)
        mean_std_all_levels[:, i_level] = mean_std

    # -------------------------------------------- #
    # save
    MEAN_values[varname] = np.copy(mean_std_all_levels[0, :])
    STD_values[varname] = np.copy(mean_std_all_levels[1, :])

In [6]:
# ------------------------------------------------------- #
# Initialize dataset
ds_mean_6h = xr.Dataset(coords={'bottom_top': level})

for varname, data in MEAN_values.items():
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["bottom_top",],
            coords={"bottom_top": level},
            name=varname,
        )
        ds_mean_6h[varname] = data_array
    else:
        data_array = xr.DataArray(
            data,
            name=varname,
        )
        ds_mean_6h[varname] = data_array

In [7]:
#ds_mean_6h.to_netcdf('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/C404_6h_mean_1980_2019_16lev.nc')

In [5]:
ds_mean_6h = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/C404_6h_std_1980_2019_16lev.nc')

In [6]:
ds_mean_6h['WRF_precip'].values

array(1.06347254)

In [11]:
# ------------------------------------------------------- #
# create xr.DataArray for std

# use the same level coord as mean
ds_std_6h = xr.Dataset(coords={'bottom_top': level})

for varname, data in STD_values.items():
    data = np.sqrt(data)
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["bottom_top",],
            coords={"bottom_top": level},
            name=varname,
        )
        ds_std_6h[varname] = data_array
    else:
        data_array = xr.DataArray(
            data,
            name=varname,
        )
        ds_std_6h[varname] = data_array

In [12]:
# ds_std_6h.to_netcdf('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/C404_6h_std_1980_2019_16lev.nc')

In [13]:
ds_std_6h

<xarray.Dataset>
Dimensions:              (bottom_top: 16)
Coordinates:
  * bottom_top           (bottom_top) float32 0.0 1.0 2.0 3.0 ... 13.0 14.0 15.0
Data variables: (12/18)
    WRF_TCC              float64 0.4985
    WRF_evapor           float64 0.1292
    WRF_PWAT             float64 0.01356
    WRF_U10              float64 2.379
    WRF_OLR              float64 37.89
    WRF_MLCAPE           float64 626.5
    ...                   ...
    WRF_SP               float64 3.881e+03
    WRF_U                (bottom_top) float64 2.865 4.805 5.694 ... 12.5 8.632
    WRF_V                (bottom_top) float64 3.85 6.716 8.01 ... 7.705 3.777
    WRF_T                (bottom_top) float64 10.44 9.971 9.686 ... 4.87 3.159
    WRF_Q                (bottom_top) float64 0.004979 0.004924 ... 2.819e-07
    WRF_P                (bottom_top) float64 3.868e+03 3.809e+03 ... 10.79

In [10]:
# import matplotlib.pyplot as plt
# %matplotlib inline

# SP = ds_example['WRF_SP'].isel(time=999).values
# plt.pcolormesh(SP)
# plt.colorbar()